# Bootstrap confidence intervals and paired significance tests

**Question answered.** Builds a reusable statistics harness: bootstrap 95% CI on macro-F1 /
balanced accuracy (resampling test *sequences*, which is what captures small-class
instability that a 3-seed standard deviation misses), McNemar's exact test for two models on
the *same* test set, Wilcoxon signed-rank across seeds, and Holm-Bonferroni correction for
multiple comparisons. This notebook demonstrates the harness on the main full-length and
fragmented benchmark, and is written so you can re-apply the same functions to the outputs
of LeakageDelta, OnehotCNN, and ToolHeadToHead.

**GPU needed.** No -- this notebook only needs the already-computed predictions (it refits
the main-paper classical + a subset of deep models to get held-out predictions, or, if you
already have saved prediction arrays from earlier runs, point `PRED_CACHE` at them instead
of refitting). If you refit the CNNs here to get their predictions, turn on GPU; if you
already have saved `.npy` prediction files from the main experiments, this whole notebook
runs on CPU in minutes.

**Runtime.** The bootstrap itself (2000 resamples x however many model/length combinations)
is CPU-only and takes seconds per combination. If refitting models from scratch, add their
usual training time (see LeakageDelta/OnehotCNN estimates).

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports

In [ ]:

import warnings; warnings.filterwarnings("ignore")
!pip install -q statsmodels
import os
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)


### 2. The reusable statistics harness (copy these functions into any other notebook)

In [ ]:

def bootstrap_ci(y_true, y_pred, metric=lambda t, p: f1_score(t, p, average="macro"),
                  n_boot=2000, seed=0):
    """Resample TEST SEQUENCES with replacement, recompute the metric each time.
    The 2.5th/97.5th percentiles are the 95% CI. This captures the small-class instability
    that a 3-seed standard deviation completely misses (e.g. one wrong subtype-G sequence
    moves macro-F1 by ~0.44 points on a ~4,500-sequence test set)."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true); stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        stats.append(metric(y_true[idx], y_pred[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(np.mean(stats)), float(lo), float(hi)

def paired_mcnemar(y_true, pred_a, pred_b):
    """Are model A and model B significantly different ON THE SAME test set?
    2x2 discordance table; only 'A right & B wrong' and 'A wrong & B right' matter.
    Exact binomial test (appropriate for small discordant counts)."""
    y_true = np.asarray(y_true)
    a_ok = (np.asarray(pred_a) == y_true); b_ok = (np.asarray(pred_b) == y_true)
    tbl = [[np.sum(a_ok & b_ok),  np.sum(a_ok & ~b_ok)],
           [np.sum(~a_ok & b_ok), np.sum(~a_ok & ~b_ok)]]
    result = mcnemar(tbl, exact=True)
    return dict(statistic=result.statistic, pvalue=result.pvalue, table=tbl)

def paired_wilcoxon(scores_a, scores_b):
    """Across-seed comparison: scores_a/scores_b are equal-length arrays of a metric
    (e.g. macro-F1) computed on the SAME seeds/splits for model A and model B."""
    stat, p = wilcoxon(scores_a, scores_b)
    return dict(statistic=stat, pvalue=p)

def holm_bonferroni(pvalues, alpha=0.05):
    """Holm-Bonferroni step-down correction. Returns adjusted-significance booleans in
    the ORIGINAL order of `pvalues`."""
    pvalues = np.asarray(pvalues)
    order = np.argsort(pvalues)
    m = len(pvalues)
    reject = np.zeros(m, dtype=bool)
    for rank, idx in enumerate(order):
        thresh = alpha / (m - rank)
        if pvalues[idx] <= thresh:
            reject[idx] = True
        else:
            break  # Holm's step-down stops at the first non-rejection
    return reject


### 3. Load data and fit the models you want CIs for (edit `MODELS_TO_RUN` as needed)

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
grp_all = df["cluster"].values

SEED = 0
s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trv, te = next(s1.split(X_all, y_all, grp_all))
ytr, yte = y_all[trv], y_all[te]

vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=512, lowercase=False)
Xtr = vec.fit_transform(X_all[trv]); Xte = vec.transform(X_all[te])

MODELS_TO_RUN = {
    "svm": LinearSVC(class_weight="balanced", random_state=SEED, max_iter=20000),
    "rf":  RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1),
}
predictions = {}
for name, clf in MODELS_TO_RUN.items():
    clf.fit(Xtr, ytr)
    predictions[name] = clf.predict(Xte)
    print(name, "macro-F1:", f1_score(yte, predictions[name], average="macro"))

# If you already have prediction arrays saved from LeakageDelta/OnehotCNN/ToolHeadToHead (e.g. np.save'd .npy files or
# columns in a results CSV), load them here instead of refitting:
#   predictions["resnet18_300bp"] = np.load("/kaggle/input/.../resnet18_300bp_preds.npy")


### 4. Bootstrap CIs for every model

In [ ]:

ci_rows = []
for name, pred in predictions.items():
    mean_f1, lo_f1, hi_f1 = bootstrap_ci(yte, pred, metric=lambda t,p: f1_score(t,p,average="macro"))
    mean_ba, lo_ba, hi_ba = bootstrap_ci(yte, pred, metric=balanced_accuracy_score)
    ci_rows.append(dict(model=name, f1_mean=mean_f1, f1_ci_lo=lo_f1, f1_ci_hi=hi_f1,
                         bal_acc_mean=mean_ba, bal_acc_ci_lo=lo_ba, bal_acc_ci_hi=hi_ba))
ci_table = pd.DataFrame(ci_rows).round(4)
ci_table.to_csv(f"{OUT}/bootstrapsig_bootstrap_ci_table.csv", index=False)
ci_table


### 5. Pairwise McNemar tests between every pair of models, Holm-Bonferroni corrected

In [ ]:

from itertools import combinations

pair_rows = []
names = list(predictions.keys())
for a, b in combinations(names, 2):
    res = paired_mcnemar(yte, predictions[a], predictions[b])
    pair_rows.append(dict(model_a=a, model_b=b, statistic=res["statistic"], pvalue=res["pvalue"]))

pair_df = pd.DataFrame(pair_rows)
pair_df["reject_holm_bonferroni_0.05"] = holm_bonferroni(pair_df["pvalue"].values, alpha=0.05)
pair_df.to_csv(f"{OUT}/bootstrapsig_pairwise_mcnemar.csv", index=False)
pair_df


### 6. Across-seed Wilcoxon comparison (template — fill `scores_a` / `scores_b` from LeakageDelta's/OnehotCNN's multi-seed CSVs)

In [ ]:

# Example: compare linear SVM vs ResNet-18 macro-F1 across 5 seeds using a results CSV
# produced by LeakageDelta or the main fragmentation sweep, with columns model/seed/f1.
example_path = f"{OUT}/leakagedelta_leakage_delta_raw.csv"
if os.path.exists(example_path):
    raw = pd.read_csv(example_path)
    sub = raw[raw["arm"] == "cluster_disjoint"]
    piv = sub.pivot_table(index="seed", columns="model", values="f1")
    if {"svm", "resnet18"}.issubset(piv.columns):
        res = paired_wilcoxon(piv["svm"].values, piv["resnet18"].values)
        print("SVM vs ResNet-18 across seeds (Wilcoxon):", res)
else:
    print(f"{example_path} not found -- run LeakageDelta first, or point this cell at any results CSV "
          "with columns [model, seed, f1] from another notebook.")


### 7. How every headline number in the paper should now be reported

In [ ]:

print("""
Going forward, every macro-F1/balanced-accuracy number in the manuscript should be reported as:
    mean [95% CI: lo-hi]  (bootstrap over test sequences, n_boot=2000)
and every 'model X beats model Y' claim should carry:
    McNemar exact test, p = ..., Holm-Bonferroni-corrected across the ~15 pairwise
    comparisons made per fragment length.
Where CIs overlap for two models, the paper should say so explicitly and not declare a winner.
""")


### 8. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r bootstrapsig_statistics_outputs.zip bootstrapsig_bootstrap_ci_table.csv bootstrapsig_pairwise_mcnemar.csv